In [12]:
from playwright.async_api import async_playwright
import time
import os
async def login():
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context()
        page = await context.new_page()

        await page.goto("https://www.naukri.com/")
        print("Login manually in this window...")

        input("Press ENTER after login is complete...")

        await context.storage_state(path="naukri_state.json")
        
apply = await login()

Login manually in this window...


In [ ]:
async def scrape_jobs():
    jobs_data = []
    category = "data-science-internship"  # Change this to your desired category

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context(storage_state="state.json")
        
        page = await context.new_page()
        # search_url = f"https://internshala.com/internships/{category}"
        # await page.goto(
        #     "https://internshala.com/internships/"+category, 
        #     wait_until="networkidle"
        # )

In [2]:
import asyncio
from urllib.parse import quote
from playwright.async_api import async_playwright


from urllib.parse import quote


def construct_naukri_url(job_titles: list[str], experience: int, role_type: str) -> str:
    """
    Build Naukri search URL correctly for job or internship.
    """

    role_type = role_type.lower().strip()

    # Clean titles
    titles = [t.strip().lower() for t in job_titles]

    # -------- PATH PART --------
    # hyphen separated
    slug_part = "-".join(t.replace(" ", "-") for t in titles)

    if role_type == "internship":
        path = f"{slug_part}-internship-jobs"
    else:
        path = f"{slug_part}-jobs"

    # -------- QUERY PART --------
    # comma separated
    keyword_string = ", ".join(titles)

    if role_type == "internship":
        keyword_string += " internship"

    encoded_keywords = quote(keyword_string)

    # -------- FINAL URL --------
    base = f"https://www.naukri.com/{path}"

    if role_type == "internship":
        url = (
            f"{base}"
            f"?k={encoded_keywords}"
            f"&experience={experience}"
            f"&qproductJobSource=2"
            f"&qinternshipFlag=true"
            f"&naukriCampus=true"
        )
    else:
        url = (
            f"{base}"
            f"?k={encoded_keywords}"
            f"&experience={experience}"
            f"&qproductJobSource=2"
            f"&naukriCampus=true"
        )

    return url

import asyncio
from playwright.async_api import async_playwright


async def scrape_naukri_jobs(page):
    jobs_data = []

    # Wait until job cards load
    await page.wait_for_selector(".srp-jobtuple-wrapper", timeout=15000)

    cards = await page.query_selector_all(".srp-jobtuple-wrapper")

    for card in cards:
        try:
            # Title
            title_el = await card.query_selector("h2 a.title")
            title = await title_el.inner_text() if title_el else None
            link = await title_el.get_attribute("href") if title_el else None

            # Company
            company_el = await card.query_selector(".comp-name")
            company = await company_el.inner_text() if company_el else None

            # Experience (NOT duration for jobs)
            exp_el = await card.query_selector(".exp-wrap span span")
            experience = await exp_el.inner_text() if exp_el else None

            # Salary / Stipend
            sal_el = await card.query_selector(".sal-wrap span span")
            stipend = await sal_el.inner_text() if sal_el else None

            # Location
            loc_el = await card.query_selector(".loc-wrap span span")
            location = await loc_el.inner_text() if loc_el else None

            # Posted / Starts in
            posted_el = await card.query_selector(".job-post-day")
            starts_in = await posted_el.inner_text() if posted_el else None

            # Description (NOT present in listing card)
            # Naukri listing page doesn't contain full description in card.
            description = None

            jobs_data.append({
                "title": title,
                "link": link,
                "company": company,
                "location": location,
                "salary_or_stipend": stipend,
                "experience_or_duration": experience,
                "description": description,
                "starts_in": starts_in
            })

        except Exception as e:
            print("Error parsing card:", e)

    return jobs_data

async def run_naukri_search(job_titles, experience, role_type):
    constructed_urls = []

    # Build single combined URL
    url = construct_naukri_url(job_titles, experience, role_type)
    constructed_urls.append(url)

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context(storage_state="naukri_state.json")
        page = await context.new_page()

        # Navigate to constructed search URL
        await page.goto(url, wait_until="domcontentloaded")
        print("Navigated to:", url)
        global jobs
        jobs = await scrape_naukri_jobs(page)
        print(f"Fetched {len(jobs)} jobs")
        for job in jobs:
            print(job)

        await browser.close()
        
        await asyncio.sleep(5)  # Let page load visually
        await browser.close()

    return constructed_urls

# Preffered job titles. Primary job role and other preffered job roles that are typed during the onboarding will be added here.
job_titles = ["ai ml engineer", "gen ai developer", "data scientist"]
experience = 0
role_type = "job"  # or "job"

await run_naukri_search(job_titles, experience, role_type)

Navigated to: https://www.naukri.com/ai-ml-engineer-gen-ai-developer-data-scientist-jobs?k=ai%20ml%20engineer%2C%20gen%20ai%20developer%2C%20data%20scientist&experience=0&qproductJobSource=2&naukriCampus=true
Fetched 20 jobs
{'title': 'ML / Data Scientist / Gen AI Specialist', 'link': 'https://www.naukri.com/job-listings-ml-data-scientist-gen-ai-specialist-gibots-pune-0-to-4-years-030724500606', 'company': 'Gibots', 'location': 'Pune', 'salary_or_stipend': None, 'experience_or_duration': '0-4 Yrs', 'description': None, 'starts_in': '3+ weeks ago'}
{'title': 'Gen AI Engineer', 'link': 'https://www.naukri.com/job-listings-gen-ai-engineer-exotel-bengaluru-0-to-2-years-270126015270', 'company': 'Exotel', 'location': 'Bengaluru', 'salary_or_stipend': '5-15 Lacs PA', 'experience_or_duration': '0-2 Yrs', 'description': None, 'starts_in': '3+ weeks ago'}
{'title': 'Data Scientist', 'link': 'https://www.naukri.com/job-listings-data-scientist-persolkelly-india-gurugram-0-to-1-years-200126014641'

['https://www.naukri.com/ai-ml-engineer-gen-ai-developer-data-scientist-jobs?k=ai%20ml%20engineer%2C%20gen%20ai%20developer%2C%20data%20scientist&experience=0&qproductJobSource=2&naukriCampus=true']

In [4]:
jobs

[{'title': 'Junior AI/ML Engineer ( Fresher)',
  'link': 'https://www.naukri.com/job-listings-junior-ai-ml-engineer-fresher-reizend-pvt-ltd-thiruvananthapuram-0-to-1-years-110226038080',
  'company': 'Reizend It Consultants',
  'location': 'Thiruvananthapuram',
  'salary_or_stipend': '4-6 Lacs PA',
  'experience_or_duration': '0-1 Yrs',
  'description': None,
  'starts_in': '2 weeks ago'},
 {'title': 'ML / Data Scientist / Gen AI Specialist',
  'link': 'https://www.naukri.com/job-listings-ml-data-scientist-gen-ai-specialist-gibots-pune-0-to-4-years-030724500606',
  'company': 'Gibots',
  'location': 'Pune',
  'salary_or_stipend': None,
  'experience_or_duration': '0-4 Yrs',
  'description': None,
  'starts_in': '3+ weeks ago'},
 {'title': 'Gen AI Engineer',
  'link': 'https://www.naukri.com/job-listings-gen-ai-engineer-exotel-bengaluru-0-to-2-years-270126015270',
  'company': 'Exotel',
  'location': 'Bengaluru',
  'salary_or_stipend': '5-15 Lacs PA',
  'experience_or_duration': '0-2 Yr

In [3]:
jobs[1]

{'title': 'Gen AI Engineer',
 'link': 'https://www.naukri.com/job-listings-gen-ai-engineer-exotel-bengaluru-0-to-2-years-270126015270',
 'company': 'Exotel',
 'location': 'Bengaluru',
 'salary_or_stipend': '5-15 Lacs PA',
 'experience_or_duration': '0-2 Yrs',
 'description': None,
 'starts_in': '3+ weeks ago'}

# Applying to a page

In [ ]:
jobs[3]['link']

'https://www.naukri.com/job-listings-ai-ml-engineer-eidiko-systems-integrators-hyderabad-0-to-3-years-260226017010'

In [ ]:
"""
naukri_applier.py

Handles Naukri application flows:
  CASE 1A — Apply button → immediate success message
  CASE 1B — Apply button → chatbot Q&A appears (handled automatically)
  CASE 2  — "I am interested" button → chatbot Q&A loop

Usage:
    await apply(page, job_url)

Terminal answers for now — swap _get_answer_terminal() with LLM later.
"""

import asyncio
from playwright.async_api import Page, async_playwright

# ── Selectors ──────────────────────────────────────────────────────────────────

SEL_APPLY_BTN       = "#apply-button"
SEL_ALREADY_APPLIED = "#already-applied, span.already-applied"
SEL_APPLY_SUCCESS   = "span.apply-message"

SEL_WALKIN_BTN      = "#walkin-button"
SEL_CHAT_LIST       = "ul[id^='chatList__']"
SEL_BOT_MESSAGES    = "li.botItem .botMsg span"
SEL_RADIO_CONTAINER = "div[id^='singleselect_radiobutton__']"
SEL_RADIO_INPUTS    = "input[type='radio'].ssrc__radio"
SEL_TEXT_INPUT      = "div.textArea[contenteditable='true']"
SEL_SAVE_BTN        = "div.sendMsg"


# ── Entry point ────────────────────────────────────────────────────────────────

async def apply(page: Page, job_url: str) -> None:
    """Navigate to job URL and apply via whichever flow is available."""
    print(f"\n→ Opening: {job_url}")

    # networkidle never fires on Naukri — use domcontentloaded + explicit wait
    await page.goto(job_url, wait_until="domcontentloaded", timeout=60_000)

    try:
        await page.wait_for_selector(
            f"{SEL_APPLY_BTN}, {SEL_WALKIN_BTN}, {SEL_ALREADY_APPLIED}",
            timeout=15_000
        )
    except Exception:
        print("[WARN] Timed out waiting for apply/walkin button. Proceeding anyway...")

    # Check already-applied FIRST before anything else
    if await page.locator(SEL_ALREADY_APPLIED).count() > 0:
        print("⚠️  Already applied to this job.")
        return

    apply_btn  = page.locator(SEL_APPLY_BTN).first
    walkin_btn = page.locator(SEL_WALKIN_BTN).first

    has_apply  = await apply_btn.count() > 0
    has_walkin = await walkin_btn.count() > 0

    if has_apply:
        print("[CASE 1] Apply button found.")
        await _handle_apply_button(page, apply_btn)

    elif has_walkin:
        print("[CASE 2] 'I am interested' button found.")
        await walkin_btn.click()
        await page.wait_for_selector(SEL_CHAT_LIST, timeout=10_000)
        print("[CHAT] Chatbot opened.")
        await _chatbot_loop(page)

    else:
        print("[ERROR] Neither apply nor walkin button found on this page.")


# ── Apply button handler ───────────────────────────────────────────────────────

async def _handle_apply_button(page: Page, apply_btn) -> None:
    """
    Click Apply. Can result in:
      A) Immediate success message        → done
      B) Chatbot Q&A form appears         → hand off to chatbot loop
      C) Button says 'apply on company
         website' OR page navigates away  → print external link
      D) Neither                          → warn user
    """
    original_url = page.url

    # C — check button text BEFORE clicking
    btn_text = (await apply_btn.text_content() or "").strip().lower()
    if "company website" in btn_text or "external" in btn_text:
        # Open in new tab so we can capture the redirected URL
        async with page.context.expect_page() as new_page_info:
            await apply_btn.click()
        new_page = await new_page_info.value
        await new_page.wait_for_load_state("domcontentloaded")
        print(f"🔗 Apply on company website — use this link to apply: {new_page.url}")
        return

    await apply_btn.click()
    print("  Apply clicked. Waiting for response...")
    await asyncio.sleep(2)

    # C — check if we got navigated to a different domain (external redirect)
    current_url = page.url
    if current_url != original_url and "naukri.com" not in current_url:
        print(f"🔗 Redirected to external site — use this link to apply: {current_url}")
        return

    try:
        await page.wait_for_selector(
            f"{SEL_APPLY_SUCCESS}, {SEL_CHAT_LIST}",
            timeout=15_000
        )
    except Exception:
        # Last check: maybe we navigated away during the wait
        if "naukri.com" not in page.url:
            print(f"🔗 Redirected to external site — use this link to apply: {page.url}")
        else:
            print("[WARN] Nothing detected 15s after Apply click. Check the browser manually.")
        return

    # A) Instant success
    if await page.locator(SEL_APPLY_SUCCESS).count() > 0:
        msg = (await page.locator(SEL_APPLY_SUCCESS).first.text_content() or "").strip()
        print(f"✓ Successfully applied! {msg}")
        return

    # B) Chatbot appeared
    if await page.locator(SEL_CHAT_LIST).count() > 0:
        print("[CASE 1→CHAT] Chatbot appeared after Apply. Switching to chat flow...")
        await _chatbot_loop(page)
        return

    print("[WARN] Unexpected state after Apply click.")


# ── Shared chatbot Q&A loop ────────────────────────────────────────────────────

async def _chatbot_loop(page: Page) -> None:
    """
    Core Q&A loop — works for both CASE 1B and CASE 2.
    Reads each bot question, asks user for answer, fills and submits.
    Exits on success message or after too many idle iterations.
    """
    seen_bot_messages: set[str] = set()
    idle = 0
    max_idle = 10

    while True:
        await asyncio.sleep(2)

        # ── Success? ────────────────────────────────────────────────────────
        if await page.locator(SEL_APPLY_SUCCESS).count() > 0:
            msg = (await page.locator(SEL_APPLY_SUCCESS).first.text_content() or "").strip()
            print(f"\n✓ Done! {msg}")
            return

        # ── Radio question ──────────────────────────────────────────────────
        radio_container = page.locator(SEL_RADIO_CONTAINER).first
        if await radio_container.count() > 0:
            question_text = await _get_latest_bot_message(page, seen_bot_messages)
            if question_text:
                seen_bot_messages.add(question_text)
            options = await _get_radio_options(radio_container)
            answer = await _get_answer_terminal(question_text or "Choose an option:", options)
            await _click_radio_option(radio_container, answer)
            await _click_save(page)
            idle = 0
            continue

        # ── Text input question ─────────────────────────────────────────────
        input_el = page.locator(SEL_TEXT_INPUT).first
        if await input_el.count() > 0:
            question_text = await _get_latest_bot_message(page, seen_bot_messages)
            if not question_text:
                await asyncio.sleep(1)
                continue
            seen_bot_messages.add(question_text)
            answer = await _get_answer_terminal(question_text, options=None)
            await _type_in_input(page, input_el, answer)
            await _click_save(page)
            idle = 0
            continue

        # ── Idle guard ──────────────────────────────────────────────────────
        idle += 1
        if idle >= max_idle:
            print("[WARN] No actionable element found. Stopping chat loop.")
            return
        print(f"[WAIT] Waiting for next question... ({idle}/{max_idle})")


# ── Helpers ────────────────────────────────────────────────────────────────────

async def _get_latest_bot_message(page: Page, seen: set[str]) -> str | None:
    """Return the most recent unseen bot message."""
    spans = page.locator(SEL_BOT_MESSAGES)
    count = await spans.count()
    for i in range(count - 1, -1, -1):
        text = (await spans.nth(i).text_content() or "").strip()
        if text and text not in seen:
            return text
    return None


async def _get_radio_options(container) -> list[str]:
    inputs = container.locator(SEL_RADIO_INPUTS)
    count  = await inputs.count()
    options = []
    for i in range(count):
        val = await inputs.nth(i).get_attribute("value") or ""
        options.append(val)
    return options


async def _click_radio_option(container, answer: str) -> None:
    """
    Click radio by matching value against answer (case-insensitive).
    Uses the <label> element to click — avoids 'outside viewport' errors.
    Falls back to JS click if label click also fails.
    """
    inputs = container.locator(SEL_RADIO_INPUTS)
    count  = await inputs.count()

    for i in range(count):
        radio = inputs.nth(i)
        val   = (await radio.get_attribute("value") or "").strip().lower()

        if val == answer.strip().lower():
            radio_id = await radio.get_attribute("id") or ""

            # 1st attempt: click the <label for="..."> — it's always visible
            if radio_id:
                label = container.locator(f"label[for='{radio_id}']").first
                if await label.count() > 0:
                    await label.scroll_into_view_if_needed()
                    await label.click()
                    print(f"  ✓ Selected (label click): {answer}")
                    return

            # 2nd attempt: JS click — bypasses viewport check entirely
            await radio.evaluate("el => el.click()")
            print(f"  ✓ Selected (JS click): {answer}")
            return

    print(f"  [WARN] Radio option '{answer}' not found. Available: {[await inputs.nth(i).get_attribute('value') for i in range(count)]}")


async def _type_in_input(page: Page, input_el, answer: str) -> None:
    """Type into contenteditable div (doesn't support .fill())."""
    await input_el.click()
    await page.keyboard.press("Control+a")
    await page.keyboard.type(answer)
    print(f"  ✓ Typed: {answer}")


async def _click_save(page: Page) -> None:
    save_btn = page.locator(SEL_SAVE_BTN).first
    await save_btn.wait_for(state="visible", timeout=5_000)
    await save_btn.click()
    print("  → Saved.\n")
    await asyncio.sleep(2)


# ── Terminal input (replace with LLM later) ───────────────────────────────────

async def _get_answer_terminal(question: str, options: list[str] | None) -> str:
    print("\n" + "─" * 60)
    print(f"QUESTION: {question}")
    if options:
        print("OPTIONS:")
        for i, opt in enumerate(options, 1):
            print(f"  {i}. {opt}")
        print("(Type exact option text or its number)")

    answer = await asyncio.get_event_loop().run_in_executor(
        None, lambda: input("YOUR ANSWER: ").strip()
    )

    if options and answer.isdigit():
        idx = int(answer) - 1
        if 0 <= idx < len(options):
            answer = options[idx]
            print(f"  → Resolved to: {answer}")

    return answer



# ── Standalone runner ──────────────────────────────────────────────────────────


async def main():
    JOB_URL = jobs[8]['link']

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        # Load saved session if available, otherwise fresh context
        try:
            context = await browser.new_context(storage_state="naukri_state.json")
            print("[AUTH] Loaded saved session.")
        except Exception:
            context = await browser.new_context()
            print("[AUTH] No saved session found — you may need to log in manually.")

        page = await context.new_page()
        await apply(page, JOB_URL)

        input("\nPress ENTER to close browser...")
        await browser.close()

await main()
        


[AUTH] Loaded saved session.

→ Opening: https://www.naukri.com/job-listings-pci-engineer-yokogawa-electric-international-pte-bengaluru-0-to-7-years-260226503023
[WARN] Timed out waiting for apply/walkin button. Proceeding anyway...
[ERROR] Neither apply nor walkin button found on this page.


In [26]:
# 8 apply on company site

async def main():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        context = await browser.new_context(storage_state="naukri_state.json")
        page = await context.new_page()

        await page.goto(jobs[8]['link'])
        input()
        
await main()